In [1]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("Hf-token")


In [2]:
# %% CELL 1 — installs (same as your notebooks)
!pip uninstall -y mamba-ssm causal-conv1d -q
!TORCH_CUDA_ARCH_LIST="7.5" pip install causal-conv1d --no-build-isolation --no-cache-dir -q
!TORCH_CUDA_ARCH_LIST="7.5" pip install mamba-ssm --no-build-isolation --no-cache-dir -q
!pip install transformers -q


  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.4/216.4 kB 3.9 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (pyproject.toml) ... done


In [ ]:
# %% CELL 2 — imports & config
import os, json, random, math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from transformers import AutoTokenizer
from mamba_ssm import Mamba

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

IMG_DIR = "/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/train2017/"
ANN_FILE = "/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/annotations/captions_train2017.json"
TEXT_CKPT = "/kaggle/input/datasets/mehtasanjana/text-checkpoint/latest_model.pt"  

CHECKPOINT_DIR = "/kaggle/working/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

SHARED_DIM  = 256     # size of the joint contrastive embedding space
BATCH_SIZE  = 64
EPOCHS      = 5
LR          = 3e-4
MAX_CAP_LEN = 64

In [ ]:
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}:", torch.cuda.get_device_name(i))

print("DEVICE:", DEVICE)

In [ ]:
# %% CELL 3 — VisionMambaEncoder 
class VisionMambaEncoder(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_chans=3, embed_dim=384, depth=8):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches + 1, embed_dim))
        self.layers = nn.ModuleList([
            nn.ModuleDict({
                'norm': nn.LayerNorm(embed_dim),
                'mamba_fwd': Mamba(d_model=embed_dim, d_state=16, d_conv=4, expand=2),
                'mamba_bwd': Mamba(d_model=embed_dim, d_state=16, d_conv=4, expand=2)
            }) for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.apply(self._init_weights)
        nn.init.normal_(self.pos_embed, std=0.02)
        nn.init.normal_(self.cls_token, std=0.02)

    def _init_weights(self, m):
        if isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.LayerNorm):
            nn.init.zeros_(m.bias)
            nn.init.ones_(m.weight)

    def forward(self, x):
        B = x.shape[0]
        x = self.proj(x).flatten(2).transpose(1, 2)
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)
        x = x + self.pos_embed
        for layer in self.layers:
            normed = layer['norm'](x)
            fwd = layer['mamba_fwd'](normed)
            bwd = layer['mamba_bwd'](normed.flip([1])).flip([1])
            x = x + fwd + bwd
        return self.norm(x)   # (B, num_patches+1, embed_dim) — index 0 is the cls token

In [ ]:
# %% CELL 4 — MambaLM 
try:
    from mamba_ssm import Mamba as _CudaMamba
    HAS_CUDA_MAMBA = True
except ImportError:
    HAS_CUDA_MAMBA = False

class MambaBlock(nn.Module):
    def __init__(self, d_model, d_state=16, expand=2, use_grad_checkpoint=False):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.mamba = _CudaMamba(d_model=d_model, d_state=d_state, expand=expand)

    def forward(self, x):
        return x + self.mamba(self.norm(x))

class MambaLMConfig:
    def __init__(self, vocab_size=50257, block_size=1024, d_model=384, n_layer=16,
                 d_state=16, expand=2, use_grad_checkpoint=False):
        self.vocab_size, self.block_size = vocab_size, block_size
        self.d_model, self.n_layer = d_model, n_layer
        self.d_state, self.expand = d_state, expand
        self.use_grad_checkpoint = use_grad_checkpoint

class MambaLM(nn.Module):
    def __init__(self, cfg: MambaLMConfig):
        super().__init__()
        self.cfg = cfg
        self.embed = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList([
            MambaBlock(cfg.d_model, d_state=cfg.d_state, expand=cfg.expand)
            for _ in range(cfg.n_layer)
        ])
        self.norm_out = nn.LayerNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.embed.weight

    def hidden_states(self, input_ids):
        """Returns per-token hidden states BEFORE the lm_head — this is what we
        pool for the text embedding. (Captioning code only ever used lm_head;
        we skip it entirely here.)"""
        x = self.embed(input_ids)
        for blk in self.blocks:
            x = blk(x)
        return self.norm_out(x)   # (B, seq_len, d_model)

In [ ]:
# %% CELL 5 — Projection heads + the vision-only CLIMP wrapper
class ProjectionHead(nn.Module):
    def __init__(self, in_dim, out_dim=SHARED_DIM):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, out_dim), nn.GELU(), nn.Linear(out_dim, out_dim))
    def forward(self, x):
        return F.normalize(self.net(x), dim=-1)

class ClimpVisionTrainer(nn.Module):
    def __init__(self, vision_encoder, text_encoder, vision_dim, text_dim):
        super().__init__()
        self.vision_encoder = vision_encoder          # TRAINABLE
        self.text_encoder = text_encoder               # FROZEN
        self.vision_proj = ProjectionHead(vision_dim)   # TRAINABLE
        self.text_proj = ProjectionHead(text_dim)        # TRAINABLE (small, cheap)
        self.logit_scale = nn.Parameter(torch.tensor(2.6592))  # ln(14), CLIP-style init

    def encode_image(self, images):
        feats = self.vision_encoder(images)      # (B, num_patches+1, dim)
        cls_feat = feats[:, 0, :]                  # cls token = pooled image representation
        return self.vision_proj(cls_feat)

    def encode_text(self, token_ids, attn_mask):
        with torch.no_grad():                       # text tower frozen — no grad, saves memory
            hidden = self.text_encoder.hidden_states(token_ids)   # (B, seq, dim)
        lengths = attn_mask.sum(dim=1) - 1           # index of last real (non-pad) token
        last_tok = hidden[torch.arange(hidden.size(0)), lengths]  # (B, dim)
        return self.text_proj(last_tok)

    def forward(self, images, token_ids, attn_mask):
        img_emb = self.encode_image(images)
        txt_emb = self.encode_text(token_ids, attn_mask)
        scale = self.logit_scale.exp()
        return scale * img_emb @ txt_emb.t()   # (B, B) similarity matrix

In [ ]:
# %% CELL 6 — InfoNCE (symmetric contrastive) loss
def climp_loss(logits):
    B = logits.shape[0]
    targets = torch.arange(B, device=logits.device)
    loss_i2t = F.cross_entropy(logits, targets)
    loss_t2i = F.cross_entropy(logits.t(), targets)
    return (loss_i2t + loss_t2i) / 2

In [ ]:
# %% CELL 7 — COCO paired dataset (one random caption per image, GPT-2 tokenizer)
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class CocoPairDataset(Dataset):
    def __init__(self, img_dir, ann_file, transform):
        with open(ann_file, "r") as f:
            data = json.load(f)
        img_id_to_file = {im["id"]: im["file_name"] for im in data["images"]}
        caps_by_img = {}
        for ann in data["annotations"]:
            caps_by_img.setdefault(ann["image_id"], []).append(ann["caption"])
        self.samples = [(os.path.join(img_dir, img_id_to_file[i]), caps)
                         for i, caps in caps_by_img.items() if i in img_id_to_file]
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, captions = self.samples[idx]
        caption = random.choice(captions)
        image = self.transform(Image.open(img_path).convert("RGB"))
        enc = tokenizer(caption, truncation=True, max_length=MAX_CAP_LEN, return_tensors="pt")
        return image, enc["input_ids"].squeeze(0), enc["attention_mask"].squeeze(0)

def collate_fn(batch):
    images, ids_list, mask_list = zip(*batch)
    images = torch.stack(images)
    max_len = max(t.size(0) for t in ids_list)
    ids = torch.full((len(ids_list), max_len), tokenizer.pad_token_id, dtype=torch.long)
    mask = torch.zeros((len(ids_list), max_len), dtype=torch.long)
    for i, (t, m) in enumerate(zip(ids_list, mask_list)):
        ids[i, :t.size(0)] = t
        mask[i, :m.size(0)] = m
    return images, ids, mask

dataset = CocoPairDataset(IMG_DIR, ANN_FILE, transform)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                     collate_fn=collate_fn, num_workers=2, drop_last=True)

In [ ]:
# %% CELL 8 — build model, load your pretrained text checkpoint, freeze it
vision_encoder = VisionMambaEncoder(embed_dim=384, depth=8)

ckpt = torch.load(TEXT_CKPT, map_location=DEVICE)

text_cfg = MambaLMConfig(
    vocab_size=50257,
    block_size=1024,
    d_model=384,
    n_layer=16,
    d_state=16,
    expand=2,
)

text_encoder = MambaLM(text_cfg)
missing, unexpected = text_encoder.load_state_dict(ckpt, strict=False)
print("Missing:", missing)
print("Unexpected:", unexpected)

text_encoder = MambaLM(text_cfg)
missing, unexpected = text_encoder.load_state_dict(ckpt, strict=False)
print("Missing:", missing)
print("Unexpected:", unexpected)

model = ClimpVisionTrainer(vision_encoder, text_encoder,
                            vision_dim=384, text_dim=text_cfg.d_model).to(DEVICE)

for p in model.text_encoder.parameters():
    p.requires_grad = False
model.text_encoder.eval()

trainable_params = [p for p in model.parameters() if p.requires_grad]
n_trainable = sum(p.numel() for p in trainable_params)
print(f"Training {n_trainable/1e6:.1f}M params (vision encoder + both projection heads)")

optimizer = torch.optim.AdamW(trainable_params, lr=LR, weight_decay=0.05)
scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE == "cuda"))

In [ ]:
# %% CELL 9 — training loop
for epoch in range(EPOCHS):
    model.train()
    model.text_encoder.eval()   # keep frozen tower in eval mode even after model.train()
    running_loss = 0.0

    for step, (images, ids, mask) in enumerate(loader):
        images, ids, mask = images.to(DEVICE), ids.to(DEVICE), mask.to(DEVICE)

        optimizer.zero_grad()
        with torch.amp.autocast("cuda", dtype=torch.float16, enabled=(DEVICE == "cuda")):
            logits = model(images, ids, mask)
            loss = climp_loss(logits)

        if torch.isnan(loss):
            print(f"NaN loss at epoch {epoch} step {step} — skipping batch")
            optimizer.zero_grad()
            continue

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        with torch.no_grad():
            model.logit_scale.clamp_(0, math.log(100))

        running_loss += loss.item()
        if step % 50 == 0:
            with torch.no_grad():
                acc = (logits.argmax(dim=1) == torch.arange(logits.size(0), device=DEVICE)).float().mean()
            print(f"epoch {epoch} step {step}/{len(loader)} loss {loss.item():.4f} "
                  f"in-batch img->text acc {acc.item():.3f}")

    print(f"== epoch {epoch} avg_loss {running_loss/len(loader):.4f} ==")
    torch.save({
        "vision_encoder": model.vision_encoder.state_dict(),
        "vision_proj": model.vision_proj.state_dict(),
        "text_proj": model.text_proj.state_dict(),
        "logit_scale": model.logit_scale,
        "epoch": epoch,
    }, os.path.join(CHECKPOINT_DIR, f"climp_vision_epoch{epoch}.pt"))

print("Done. Vision-tower + projection-head checkpoints saved to", CHECKPOINT_DIR)

In [2]:
import shutil
import os
CHECKPOINT_DIR = "/kaggle/working/checkpoints"
src = os.path.join(CHECKPOINT_DIR, "climp_vision_epoch3.pt")
dst = os.path.join(os.getcwd(), "climp_vision_epoch3.pt")

shutil.copy2(src, dst)
print("Copied to:", dst)

Copied to: /kaggle/working/climp_vision_epoch3.pt
